STEP3 랜더러 만들기

In [1]:
!pip -q install lxml xmltodict 

In [3]:
# 기존 함수 불러오기
### STEP1 함수 호출
from pathlib import Path
from zipfile import ZipFile

def extract_hwpx(hwpx_path: str | Path, output_dir: str | Path) -> Path:
    """한글 파일 압축 푸는 함수"""
    hwpx_path = Path(hwpx_path)
    output_dir = Path(output_dir)
    
    if not hwpx_path.exists():
        raise FileNotFoundError(f"hwpx file not found: {hwpx_path}")
    
    with ZipFile(hwpx_path, "r") as zip_file:
        zip_file.extractall(output_dir)
    return output_dir

def hwpx_xml_to_json(xml_dir, save_path):
    """ection0.xml, header.xml, settings.xml-> *.json"""
    from pathlib import Path
    import xml.etree.ElementTree as ET
    import json

    xml_dir = Path(xml_dir)

    targets = {
        "section": xml_dir / "Contents" / "section0.xml",
        "header": xml_dir / "Contents" / "header.xml",
        "settings": xml_dir / "settings.xml",
    }

    def strip_ns(tag):
        return tag.split("}", 1)[-1] if "}" in tag else tag

    def xml_to_dict(elem):
        return {
            "tag": strip_ns(elem.tag),
            "attrs": elem.attrib,
            "text": (elem.text or "").strip(),
            "children": [xml_to_dict(child) for child in elem]
        }

    hwpx_jsonb = {}

    for name, path in targets.items():
        tree = ET.parse(path)
        root = tree.getroot()

        hwpx_jsonb[name] = {
            "source_path": str(path),
            "root_tag": strip_ns(root.tag),
            "data": xml_to_dict(root)
        }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(hwpx_jsonb, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {save_path}")

    return hwpx_jsonb

def json_to_hwpx(json_data, template_dir, output_hwpx):
    """json을 xml로 변환 후, 한글템플릿과 치환하여 한글 파일로 변경하는 함수"""
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    import xml.etree.ElementTree as ET

    def json_to_xml_elem(data):
        elem = ET.Element(data["tag"], data.get("attrs", {}))

        text = data.get("text", "")
        if text:
            elem.text = text

        for child in data.get("children", []):
            elem.append(json_to_xml_elem(child))

        return elem

    template_dir = Path(template_dir)

    targets = {
        "section": template_dir / "Contents" / "section0.xml",
        "header": template_dir / "Contents" / "header.xml",
        "settings": template_dir / "settings.xml",
    }

    for key, xml_path in targets.items():
        root_data = json_data[key]["data"]
        root_elem = json_to_xml_elem(root_data)

        tree = ET.ElementTree(root_elem)
        ET.indent(tree, space="  ", level=0)
        tree.write(
            xml_path,
            encoding="utf-8",
            xml_declaration=True
        )

        print("XML 치환 완료:", xml_path)

    with ZipFile(output_hwpx, "w", ZIP_DEFLATED) as z:
        for file in template_dir.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(template_dir))

    print("HWPX 생성 완료:", output_hwpx)

    return output_hwpx

def xml_to_hwpx(xml_dir, template_dir, output_hwpx):
    """템플릿 HWPX 구조를 유지하고 XML만 치환하여 HWPX 생성"""

    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from shutil import copytree, copy2, rmtree
    import tempfile

    xml_dir = Path(xml_dir)
    template_dir = Path(template_dir)

    with tempfile.TemporaryDirectory() as temp_dir:

        work_dir = Path(temp_dir) / "hwpx"

        # 1. 템플릿 전체 복사
        copytree(template_dir, work_dir)

        # 2. XML 치환
        replace_files = [
            "Contents/section0.xml",
            "Contents/header.xml",
            "settings.xml",
        ]

        for rel_path in replace_files:

            src = xml_dir / rel_path
            dst = work_dir / rel_path

            if src.exists():
                copy2(src, dst)
                print("XML 치환:", rel_path)

        # 3. HWPX 생성
        with ZipFile(output_hwpx, "w", ZIP_DEFLATED) as z:

            for file in work_dir.rglob("*"):
                if file.is_file():
                    z.write(
                        file,
                        file.relative_to(work_dir)
                    )

    print("HWPX 생성 완료:", output_hwpx)

    return output_hwpx
### STEP2 함수
from pathlib import Path
import json


def make_render_json(file_json_path, save_path):
    """HWPX XML JSON 파일 경로 -> 렌더링용 JSON 저장"""

    file_json_path = Path(file_json_path)
    save_path = Path(save_path)

    with open(file_json_path, "r", encoding="utf-8") as f:
        file_json = json.load(f)

    def walk(node, tag=None):
        results = []

        if tag is None or node.get("tag") == tag:
            results.append(node)

        for child in node.get("children", []):
            results.extend(walk(child, tag))

        return results

    def first_child(node, tag):
        for child in node.get("children", []):
            if child.get("tag") == tag:
                return child
        return None

    def to_int(value, default=None):
        try:
            return int(value)
        except (TypeError, ValueError):
            return default

    def get_text_from_run(run):
        texts = []
        for child in run.get("children", []):
            if child.get("tag") == "t":
                texts.append(child.get("text", ""))
        return "".join(texts)

    header_root = file_json["header"]["data"]
    section_root = file_json["section"]["data"]
    settings_root = file_json["settings"]["data"]

    font_map = {}
    for font in walk(header_root, "font"):
        attrs = font.get("attrs", {})
        font_id = attrs.get("id")
        face = attrs.get("face")

        if font_id is not None and face:
            font_map[font_id] = face

    char_style_map = {}

    for char_pr in walk(header_root, "charPr"):
        attrs = char_pr.get("attrs", {})
        char_id = attrs.get("id")

        font_ref = first_child(char_pr, "fontRef")
        font_id = None

        if font_ref:
            font_id = font_ref.get("attrs", {}).get("hangul")

        italic_node = first_child(char_pr, "italic")
        bold_node = first_child(char_pr, "bold")
        underline_node = first_child(char_pr, "underline")
        strikeout_node = first_child(char_pr, "strikeout")

        height = to_int(attrs.get("height"))

        char_style_map[char_id] = {
            "id": char_id,
            "font_id": font_id,
            "font": font_map.get(font_id),
            "height": height,
            "size_px_guess": height // 100 if height is not None else None,
            "textColor": attrs.get("textColor"),
            "shadeColor": attrs.get("shadeColor"),
            "bold": bold_node is not None,
            "italic": italic_node is not None,
            "underline": underline_node.get("attrs", {}) if underline_node else None,
            "strikeout": strikeout_node.get("attrs", {}) if strikeout_node else None,
            "raw_attrs": attrs,
        }

    para_style_map = {}

    for para_pr in walk(header_root, "paraPr"):
        attrs = para_pr.get("attrs", {})
        para_id = attrs.get("id")

        align = None
        heading = None
        line_spacing = None

        for child in para_pr.get("children", []):
            tag = child.get("tag")
            cattrs = child.get("attrs", {})

            if tag == "align":
                align = cattrs
            elif tag == "heading":
                heading = cattrs
            elif tag == "lineSpacing":
                line_spacing = cattrs

        para_style_map[para_id] = {
            "id": para_id,
            "align": align,
            "heading": heading,
            "lineSpacing": line_spacing,
            "raw_attrs": attrs,
        }

    paragraphs = []

    for p_index, p in enumerate(section_root.get("children", [])):
        if p.get("tag") != "p":
            continue

        p_attrs = p.get("attrs", {})
        para_id = p_attrs.get("paraPrIDRef")

        runs = []
        layout = None

        for child in p.get("children", []):
            if child.get("tag") == "run":
                run_attrs = child.get("attrs", {})
                char_id = run_attrs.get("charPrIDRef")
                text = get_text_from_run(child)

                if text:
                    runs.append({
                        "type": "text_run",
                        "text": text,
                        "charPrIDRef": char_id,
                        "style": char_style_map.get(char_id),
                        "raw_attrs": run_attrs,
                    })

            elif child.get("tag") == "linesegarray":
                linesegs = []

                for lineseg in child.get("children", []):
                    if lineseg.get("tag") == "lineseg":
                        a = lineseg.get("attrs", {})
                        linesegs.append({
                            "textpos": to_int(a.get("textpos")),
                            "vertpos": to_int(a.get("vertpos")),
                            "vertsize": to_int(a.get("vertsize")),
                            "textheight": to_int(a.get("textheight")),
                            "baseline": to_int(a.get("baseline")),
                            "spacing": to_int(a.get("spacing")),
                            "horzpos": to_int(a.get("horzpos")),
                            "horzsize": to_int(a.get("horzsize")),
                            "flags": to_int(a.get("flags")),
                            "raw_attrs": a,
                        })

                layout = {
                    "linesegs": linesegs
                }

        paragraphs.append({
            "type": "paragraph",
            "index": p_index,
            "paraPrIDRef": para_id,
            "styleIDRef": p_attrs.get("styleIDRef"),
            "paragraph_style": para_style_map.get(para_id),
            "runs": runs,
            "layout": layout,
            "raw_attrs": p_attrs,
        })

    render_json = {
        "type": "hwpx_render_json",
        "source": {
            "section": file_json["section"].get("source_path"),
            "header": file_json["header"].get("source_path"),
            "settings": file_json["settings"].get("source_path"),
        },
        "maps": {
            "fonts": font_map,
            "char_styles": char_style_map,
            "para_styles": para_style_map,
        },
        "document": {
            "paragraphs": paragraphs,
            "settings_raw": settings_root,
        }
    }

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(render_json, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {save_path}")

    return render_json
from pathlib import Path
import json


def render_json_to_section_xml(render_json_path, save_path):
    """렌더링용 JSON 파일 경로 -> section0.xml 저장"""

    import xml.etree.ElementTree as ET

    render_json_path = Path(render_json_path)
    save_path = Path(save_path)

    with open(render_json_path, "r", encoding="utf-8") as f:
        render_json = json.load(f)

    def str_attrs(attrs):
        return {
            str(k): str(v)
            for k, v in attrs.items()
            if v is not None
        }

    root = ET.Element("sec")

    paragraphs = render_json["document"]["paragraphs"]

    for paragraph in paragraphs:
        raw_p_attrs = paragraph.get("raw_attrs", {})

        p_attrs = {
            "id": raw_p_attrs.get("id", "0"),
            "paraPrIDRef": paragraph.get("paraPrIDRef", raw_p_attrs.get("paraPrIDRef", "0")),
            "styleIDRef": paragraph.get("styleIDRef", raw_p_attrs.get("styleIDRef", "0")),
            "pageBreak": raw_p_attrs.get("pageBreak", "0"),
            "columnBreak": raw_p_attrs.get("columnBreak", "0"),
            "merged": raw_p_attrs.get("merged", "0"),
        }

        p_elem = ET.SubElement(root, "p", str_attrs(p_attrs))

        for run in paragraph.get("runs", []):
            run_attrs = run.get("raw_attrs", {})

            run_elem = ET.SubElement(
                p_elem,
                "run",
                str_attrs({
                    "charPrIDRef": run.get("charPrIDRef", run_attrs.get("charPrIDRef", "0"))
                })
            )

            text = run.get("text", "")

            if text:
                t_elem = ET.SubElement(run_elem, "t")
                t_elem.text = text

        layout = paragraph.get("layout") or {}
        linesegs = layout.get("linesegs", [])

        if linesegs:
            linesegarray_elem = ET.SubElement(p_elem, "linesegarray")

            for lineseg in linesegs:
                raw_attrs = lineseg.get("raw_attrs", {})

                lineseg_attrs = {
                    "textpos": lineseg.get("textpos", raw_attrs.get("textpos")),
                    "vertpos": lineseg.get("vertpos", raw_attrs.get("vertpos")),
                    "vertsize": lineseg.get("vertsize", raw_attrs.get("vertsize")),
                    "textheight": lineseg.get("textheight", raw_attrs.get("textheight")),
                    "baseline": lineseg.get("baseline", raw_attrs.get("baseline")),
                    "spacing": lineseg.get("spacing", raw_attrs.get("spacing")),
                    "horzpos": lineseg.get("horzpos", raw_attrs.get("horzpos")),
                    "horzsize": lineseg.get("horzsize", raw_attrs.get("horzsize")),
                    "flags": lineseg.get("flags", raw_attrs.get("flags")),
                }

                ET.SubElement(
                    linesegarray_elem,
                    "lineseg",
                    str_attrs(lineseg_attrs)
                )

    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ", level=0)

    save_path.parent.mkdir(parents=True, exist_ok=True)

    tree.write(
        save_path,
        encoding="utf-8",
        xml_declaration=True
    )

    print(f"저장 완료: {save_path}")

    return save_path


랜더링 json 파일을 받고 
그 json 파일을 frontend에서 보기 쉽게 만들기

In [ ]:
from pathlib import Path
import json
import html


def render_json_to_html(render_json_path, save_path):
    """렌더링용 JSON 파일 -> HTML 파일"""

    render_json_path = Path(render_json_path)
    save_path = Path(save_path)

    with open(render_json_path, "r", encoding="utf-8") as f:
        render_json = json.load(f)

    def get_align(paragraph):
        para_style = paragraph.get("paragraph_style") or {}
        align = para_style.get("align") or {}

        # HWPX 값이 구현마다 다를 수 있어서 흔한 값만 우선 처리
        value = (
            align.get("horizontal")
            or align.get("textAlign")
            or align.get("align")
            or align.get("type")
        )

        align_map = {
            "LEFT": "left",
            "CENTER": "center",
            "RIGHT": "right",
            "JUSTIFY": "justify",
            "BOTH": "justify",
        }

        return align_map.get(value, "left")

    def run_to_span(run):
        text = html.escape(run.get("text", ""))
        style = run.get("style") or {}

        css = []

        if style.get("font"):
            css.append(f"font-family: '{style['font']}'")

        if style.get("size_px_guess"):
            css.append(f"font-size: {style['size_px_guess']}px")

        if style.get("textColor"):
            css.append(f"color: {style['textColor']}")

        if style.get("bold"):
            css.append("font-weight: bold")

        if style.get("italic"):
            css.append("font-style: italic")

        return f'<span style="{"; ".join(css)}">{text}</span>'

    body_parts = []

    for paragraph in render_json["document"]["paragraphs"]:
        runs = paragraph.get("runs", [])

        if not runs:
            body_parts.append("<p>&nbsp;</p>")
            continue

        align = get_align(paragraph)

        spans = "".join(run_to_span(run) for run in runs)

        body_parts.append(
            f'<p style="text-align: {align}; margin: 0 0 8px 0;">{spans}</p>'
        )

    html_text = f"""<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>HWPX Render</title>
</head>
<body>
{chr(10).join(body_parts)}
</body>
</html>
"""

    save_path.parent.mkdir(parents=True, exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        f.write(html_text)

    print(f"HTML 저장 완료: {save_path}")

    return save_path

In [11]:
input_file_path = Path(r"HWPX 파일\ex_글자.hwpx")
output_root = Path("step3_output")

# 폴더 없으면 생성
output_root.mkdir(parents=True, exist_ok=True)

render_json_path = output_root / f"{input_file_path.stem}_render.json"
html_path = output_root / f"{input_file_path.stem}.html"

render_json_to_html(
    render_json_path=render_json_path,
    save_path=html_path
)

HTML 저장 완료: step3_output\ex_글자.html


WindowsPath('step3_output/ex_글자.html')

In [ ]:
# 전체 흐름
